### **Fase 0**: Import Generico

In [ ]:
import os
import sys
import importlib
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Drive già montato, salto il mount.")

sys.path.append('/content/drive/MyDrive/progetto_immunogold/scripts')
importlib.invalidate_caches()

import common
importlib.reload(common)
from common import *

print(f"Device attivo: {device}")
print(paths)
log('Notebook avviato correttamente')

### **Split del dataset:**

### Assegna ogni immagine annotata a train/val/test, a livello di immagine (non di patch), rispettando i seguenti vincoli:


*   classi con 1 sola immagine annotata: va tutta in train (il modello deve comunque vedere quella scala durante il training,
      anche se non potrà essere valutato formalmente su quella classe)
*   classi con >= 2 immagini: garantita almeno una rappresentazione
      fuori da train (val o test), bilanciando i totali complessivi



In [ ]:
import random

mappa_patch = pd.read_csv(os.path.join(paths['annotations'], 'mappa_patch.csv'))

def assegna_split(mappa_patch, seed=42):

    img_classe = mappa_patch[['id_immagine', 'classe_calibrazione']].drop_duplicates().reset_index(drop=True)

    rng = random.Random(seed)
    split = {}
    contatori = {'train': 0, 'val': 0, 'test': 0}

    for classe, gruppo in img_classe.groupby('classe_calibrazione'):
        immagini = gruppo['id_immagine'].tolist()
        rng.shuffle(immagini)
        n = len(immagini)

        if n == 1:
            assegnazioni = ['train']
        elif n == 2:
            secondo = 'val' if contatori['val'] <= contatori['test'] else 'test'
            assegnazioni = ['train', secondo]
        else:
            assegnazioni = ['val', 'test'] + ['train'] * (n - 2)
            rng.shuffle(assegnazioni)

        for img, s in zip(immagini, assegnazioni):
            split[img] = s
            contatori[s] += 1

    return split, contatori

split_dict, contatori = assegna_split(mappa_patch)

split_df = mappa_patch[['id_immagine', 'classe_calibrazione']].drop_duplicates().reset_index(drop=True)
split_df['split'] = split_df['id_immagine'].map(split_dict)

print("Totali per split:", contatori)
display(split_df.sort_values(['classe_calibrazione', 'split']))

path_split = os.path.join(paths['annotations'], 'split_immagini.csv')
split_df.to_csv(path_split, index=False)
log(f"Fase 5 - split salvato in {path_split}: {contatori}")

Totali per split: {'train': 10, 'val': 4, 'test': 3}


,id_immagine,classe_calibrazione,split
12,SA-MAG_X30k_JEM-1400 Flash_22562,classe_1,train
10,SA-MAG_X3000_JEM-1400 Flash_22441,classe_10,train
13,SA-MAG_X30k_JEM-1400 Flash_22644,classe_2,train
11,SA-MAG_X30k_JEM-1400 Flash_22426,classe_2,val
8,SA-MAG_X25k_JEM-1400 Flash_22433,classe_3,test
9,SA-MAG_X25k_JEM-1400 Flash_22438,classe_3,train
7,SA-MAG_X20k_JEM-1400 Flash_22447,classe_4,train
4,SA-MAG_X15k_JEM-1400 Flash_22427,classe_5,test
5,SA-MAG_X15k_JEM-1400 Flash_22478,classe_5,train
6,SA-MAG_X15k_JEM-1400 Flash_22598,classe_5,val


[2026-09-17 10:08] Fase 5 - split salvato in /content/drive/MyDrive/progetto_immunogold/annotations/split_immagini.csv: {'train': 10, 'val': 4, 'test': 3}


### **Logica di split**:

Per le classi con 2 immagini, si fa riferimento ai due contatori ('val') e ('test'); si va ad attingere la prima immagine sempre su 'train' mentre la seconda va nel contatore che presenta meno iterazioni.

-  Per i successivi training bisognerà mantenere sempre la stessa struttura di train/val/test (le stesse immagini devono finire nello stesso set, anche se ne aggiungiamo delle nuove)